# Case Study: City Air-Quality Sensor Network

This case study demonstrates ProvSQL’s continuous-distribution surface (see the chapter on continuous distributions) end-to-end through ProvSQL Studio (see the Studio chapter). It is the first case study driven primarily by Studio rather than `psql`: random variables benefit far more from interactive visualisation – PDFs, CDFs, mixture DAG layouts, conditional histograms, simplifier before-vs-after – than from text-mode output, and the workflow below makes the rewriter, the simplifier, the analytic and Monte-Carlo paths, and conditional inference all visible in the canvas.

## The Scenario

A municipal observatory operates a small air-quality sensor network. Sensors of three different vendors report a $PM_{2.5}$ concentration (*fine particulate matter*, i.e. airborne particles with aerodynamic diameter at most 2.5 μm, expressed in micrograms per cubic metre) on a fixed schedule. The sensors differ in calibration and noise characteristics:

- high-end units report `Normal(μ, σ)` with small σ;
- low-cost units report `Uniform[μ−δ, μ+δ]` over a small window;
- a drift-prone unit reports `Exponential(λ)` while its internal hardware self-tests cycle;
- a multi-pass aggregating unit reports `Erlang(k, λ)` over the pass count.

A reference station with a calibrated lab-grade instrument contributes deterministic readings.

Regulatory categories partition the value axis: *Good* below 12, *Moderate* between 12.1 and 35, *Unhealthy* above 35.1 (loosely following the US EPA AQI breakpoints for PM2.5 in their pre-2024 form, simplified to three tiers). Each station has a Bernoulli probability of being in calibration on a given day. A separate batch table of *historical* readings carries the same shape so cross-batch queries via `UNION ALL` are meaningful.

Your tasks:

- inspect the per-row distributions and the rewriter’s effect on threshold queries;
- compute the probability that each station’s reading exceeds an *Unhealthy* threshold, exercising the planner-hook rewrite for `WHERE reading > 35`;
- model calibration uncertainty as a Bernoulli mixture and inspect the resulting `gate_mixture` shape;
- aggregate per-district readings and watch the simplifier fold the mixture cascade;
- run conditional inference (`E[reading | reading > 35]`) and see the closed-form truncated-distribution mean against the unconditional one;
- filter on the expected value of an aggregated random variable, combine today’s and yesterday’s batches with `UNION ALL`, and compare probability methods (`'independent'` vs `'monte-carlo'` vs `'tree-decomposition'`) side by side.

## Setup

*The following cells set up the database with all the content this notebook requires; run them first, ideally on a fresh database.*

In [ ]:
DROP TABLE IF EXISTS readings CASCADE;
DROP TABLE IF EXISTS historical_readings CASCADE;
DROP TABLE IF EXISTS stations CASCADE;
DROP TABLE IF EXISTS calibration_status CASCADE;
DROP TABLE IF EXISTS categories CASCADE;

In [ ]:
-- Four monitoring stations across two districts.
DROP TABLE IF EXISTS stations CASCADE;
CREATE TABLE stations (
  id        TEXT PRIMARY KEY,
  name      TEXT NOT NULL,
  district  TEXT NOT NULL
);

In [ ]:
TRUNCATE stations;
INSERT INTO stations VALUES
  ('s1', 'City Centre',         'centre'),
  ('s2', 'Riverside Park',      'centre'),
  ('s3', 'Industrial Estate',   'east'),
  ('s4', 'Suburban Reference',  'east');

In [ ]:
SELECT add_provenance('stations');

In [ ]:
-- Per-station calibration probability (Bernoulli).  The high-end and
-- multi-pass units are well calibrated; the low-cost and drift-prone
-- ones less so; the reference station is always in-spec.
DROP TABLE IF EXISTS calibration_status CASCADE;
CREATE TABLE calibration_status (
  station_id  TEXT PRIMARY KEY REFERENCES stations(id),
  p           DOUBLE PRECISION NOT NULL
);

In [ ]:
TRUNCATE calibration_status;
INSERT INTO calibration_status VALUES
  ('s1', 0.95),
  ('s2', 0.70),
  ('s3', 0.60),
  ('s4', 1.00);

In [ ]:
SELECT add_provenance('calibration_status');

In [ ]:
-- Regulatory categories (deterministic, no provenance).
DROP TABLE IF EXISTS categories CASCADE;
CREATE TABLE categories (
  name  TEXT PRIMARY KEY,
  lo    DOUBLE PRECISION NOT NULL,
  hi    DOUBLE PRECISION NOT NULL
);

In [ ]:
TRUNCATE categories;
INSERT INTO categories VALUES
  ('Good',       0.0,    12.0),
  ('Moderate',   12.1,   35.0),
  ('Unhealthy',  35.1,   1000.0);

In [ ]:
-- Today's readings.  Five rows of pm25, one per (station, sample),
-- exercising every distribution family plus the implicit numeric cast.
DROP TABLE IF EXISTS readings CASCADE;
CREATE TABLE readings (
  id          INTEGER PRIMARY KEY,
  station_id  TEXT NOT NULL REFERENCES stations(id),
  ts          TIMESTAMP NOT NULL,
  pm25        provsql.random_variable NOT NULL
);

In [ ]:
TRUNCATE readings;
INSERT INTO readings (id, station_id, ts, pm25) VALUES
  (1, 's1', '2026-05-12 08:00', provsql.normal(28.0, 2.0)),       -- high-end Gaussian
  (2, 's2', '2026-05-12 08:00', provsql.uniform(10.0, 22.0)),     -- low-cost uniform window
  (3, 's3', '2026-05-12 08:00', provsql.exponential(0.04)),       -- drift-prone, mean = 25
  (4, 's4', '2026-05-12 08:00', 15.0),                            -- reference (implicit cast)
  (5, 's1', '2026-05-12 09:00', provsql.normal(40.0, 4.0)),       -- high-end, into Unhealthy
  (6, 's2', '2026-05-12 09:00', provsql.uniform(12.0, 24.0)),     -- low-cost, into Moderate
  (7, 's3', '2026-05-12 09:00', provsql.erlang(3, 0.1)),          -- multi-pass Erlang, mean = 30
  (8, 's4', '2026-05-12 09:00', 16.5);                            -- reference

In [ ]:
SELECT add_provenance('readings');

In [ ]:
-- Yesterday's batch, used for the UNION ALL step.  Same shape, slightly
-- different distributions (a heatwave bumped the means).
DROP TABLE IF EXISTS historical_readings CASCADE;
CREATE TABLE historical_readings (
  id          INTEGER PRIMARY KEY,
  station_id  TEXT NOT NULL REFERENCES stations(id),
  ts          TIMESTAMP NOT NULL,
  pm25        provsql.random_variable NOT NULL
);

In [ ]:
TRUNCATE historical_readings;
INSERT INTO historical_readings (id, station_id, ts, pm25) VALUES
  (1, 's1', '2026-05-11 08:00', provsql.normal(34.0, 2.5)),
  (2, 's2', '2026-05-11 08:00', provsql.uniform(15.0, 28.0)),
  (3, 's3', '2026-05-11 08:00', provsql.exponential(0.03)),
  (4, 's4', '2026-05-11 08:00', 18.0),
  (5, 's1', '2026-05-11 09:00', provsql.normal(42.0, 3.0)),
  (6, 's2', '2026-05-11 09:00', provsql.uniform(20.0, 35.0)),
  (7, 's3', '2026-05-11 09:00', provsql.erlang(3, 0.08)),
  (8, 's4', '2026-05-11 09:00', 19.5);

In [ ]:
SELECT add_provenance('historical_readings');

In [ ]:
-- A provenance mapping so the Studio eval-strip's sr_formula and
-- PROV-XML export can label leaves with station names rather than
-- raw UUIDs.
DROP TABLE IF EXISTS station_mapping;
DROP TABLE IF EXISTS station_mapping CASCADE;
CREATE TABLE station_mapping AS
  SELECT s.name AS value, r.provsql AS provenance
  FROM readings r JOIN stations s ON s.id = r.station_id;
SELECT remove_provenance('station_mapping');
CREATE INDEX ON station_mapping(provenance);

The script creates the schema below and seeds the random-variable readings via the constructors documented in the chapter on continuous distributions. It is five tables:

- `stations(id, name, district)` – four monitoring stations across two districts, provenance-tracked.
- `readings(station_id, ts, pm25 random_variable)` – one `pm25` reading per station per timestamp; the `random_variable` carries the per-station noise model (normal, uniform, exponential, erlang, or a deterministic lifted from the reference station).
- `calibration_status(station_id, p)` – Bernoulli probability that each station is in calibration on the day of interest.
- `categories(name, lo, hi)` – three regulatory categories (*Good* / *Moderate* / *Unhealthy*) keyed by their interval bounds.
- `historical_readings(...)` – same shape as `readings`, populated from yesterday’s batch.

The schema panel lists the fixture’s six relations: the four provenance-tracked tables (`stations`, `calibration_status`, `readings`, `historical_readings`) carry the purple `prov` pill, `categories` is plain, and `station_mapping` is tagged `mapping`. The `pm25` column on `readings` and `historical_readings` is flagged with a terracotta `rv` pill: a heads-up that comparison and arithmetic operators on this column are intercepted by the planner hook and lifted into provenance gates, so a query like `pm25 > 35` produces a circuit rather than a Boolean.

## Step 1: Inspect a Noisy Reading

In the Studio query box:

In [ ]:
SELECT id, ts, pm25
FROM readings
WHERE station_id = 's1'
ORDER BY ts

The result table renders `pm25` as a clickable `random_variable` cell carrying the underlying gate UUID. Click into a row’s `pm25`: Studio switches to Circuit mode and renders the `gate_rv` leaf with the distribution-kind initial in the circle (*N* for a Normal, *U* for Uniform, *Exp* for Exponential, *Erl* for Erlang). Pick *Distribution profile* from the *Distribution* group of the eval strip and click `Run`: the panel returns $\mu$ and $\sigma^2$ headline stats and an inline histogram with a PDF/CDF toggle.

The histogram is backed server-side by [`rv_histogram`](https://provsql.org/doxygen-sql/html/group__circuit__introspection.html#ga53efbd4d68e0de0ca979cf52528c63db); pinning `provsql.monte_carlo_seed` in the Config panel (under *Provenance*) makes the shape reproducible across re-runs.

## Step 2: A First Probabilistic Threshold

The $PM_{2.5}$ *Unhealthy* category begins at 35.1. Find the rows whose reading might cross it:

In [ ]:
SELECT id, station_id, ts
FROM readings
WHERE pm25 > 35

Because `pm25` is a random variable, the comparison is not a yes/no test: it stands for the *event* “this reading exceeds 35”, and ProvSQL attaches that event to each row’s provenance.

Click into a result row’s auto-added `provsql` cell. Circuit mode shows the Boolean wrapper (a `gate_times` over the row’s input token and the `gate_cmp`); the cmp’s child link reaches into the `gate_rv` from Step 1.

The eval strip’s [`probability_evaluate`](https://provsql.org/doxygen-sql/html/group__probability.html#gad377e94cb1fff57141b1950cc4169c5e) entry exposes the five compiled methods (see the chapter on probabilities). Pick `monte-carlo` and set `n = 10000`; the panel returns the probability with a Hoeffding confidence band. Pin `provsql.monte_carlo_seed = 42` in the Config panel and re-run: the result is now identical across runs. Toggle the seed back to `-1` and re-run to see the band shift between runs.

## Step 3: The Simplifier in Action

The planner hook emits the comparator as a raw `gate_cmp` regardless of what its operands look like. A *simplifier* pass then folds comparators whose answer can be decided from the operand support alone, for example, `U(10, 22) > 35` is universally false because the uniform’s upper bound is below the threshold. The fold is controlled by `provsql.simplify_on_load` (default on), which the Config panel exposes under *Provenance*.

Click into row 2’s auto-added `provsql` cell from the Step 2 result (station `s2`, `pm25 ~ U(10, 22)`). With `provsql.simplify_on_load` on, the canvas shows a single `𝟘` (zero) gate: the simplifier resolved the comparator to a constant-false leaf and dropped the whole subtree. Toggle the GUC off in the Config panel and click the cell again: the canvas now shows the raw construction shape, a `gate_times` (`⊗`) over the row’s input token `ι` and a `gate_cmp` (`>`) whose children are the `U(10, 22)` leaf and the constant `35`. Both views are semantically identical; the simplified view is what the semiring evaluators and the Monte-Carlo sampler actually consume.

## Step 4: Calibration via Mixtures

Each station has a probability of being mis-calibrated; a mis-calibrated unit over-reports by 20% (the *reading* it records is `1.2` times the true value). The corrected estimate of the true reading is therefore `pm25` with probability `p` (the station is in spec) and `pm25 / 1.2` with probability `1 - p` (the report needs to be scaled back). Express this as a Bernoulli mixture:

In [ ]:
SELECT r.id, r.station_id,
       provsql.mixture(cs.p, r.pm25, r.pm25 / 1.2) AS pm25_calibrated
FROM readings r JOIN calibration_status cs USING (station_id)
WHERE r.station_id = 's1'

Click into a result row’s `pm25_calibrated` cell. Circuit mode renders the `gate_mixture` as a `Mix` node with three labelled outgoing edges (`p` / `x` / `y`) matching the SQL constructor’s argument order: `p` points to the Bernoulli mixing probability, `x` to the in-spec arm, and `y` to the correction arm.

The same node-inspector panel exposes `Distribution profile` on the mixture root. Because station `s1` is in spec 95% of the time, the histogram is dominated by the `N(28, 2)` arm and the out-of-spec `N(23.33, 1.667)` contributes only a small left shoulder rather than a visually distinct second mode; the panel headline reflects this with a mixture mean slightly below 28. To see clear bimodality, re-run the query with a larger calibration error, e.g. replace `r.pm25 / 1.2` with `r.pm25 / 2.0` so the out-of-spec arm folds to `N(14, 1)`, well separated from the in-spec `N(28, 2)`; the two peaks then show up distinctly on the histogram even at the 95%/5% weighting.

## Step 5: Aggregation Over Random Variables

Compute average $PM_{2.5}$ per district:

In [ ]:
SELECT s.district,
       avg(r.pm25)  AS avg_pm25,
       sum(r.pm25)  AS total_pm25
FROM readings r JOIN stations s ON s.id = r.station_id
GROUP BY s.district

Click into a row’s `avg_pm25` cell. Circuit mode shows the [`avg`](https://provsql.org/doxygen-sql/html/group__random__variable__type.html#ga0243c3a6d53bdc36a89ce5131b38e504) lowering: a `gate_arith(DIV, num, denom)` over two `gate_arith(PLUS, …)` subtrees, each child a per-row `gate_mixture` produced by `rv_aggregate_semimod`. The right child of the outer division is the count of *included* rows under their per-row provenance: rows whose provenance is false contribute the additive identity to both numerator and denominator. Run *Distribution profile* on the root: the panel shows the per-district average as a tight distribution centred at the inclusion-weighted mean.

## Step 6: Conditional Inference

Re-open the filtered query from Step 2:

In [ ]:
SELECT id, station_id, ts, pm25
FROM readings
WHERE pm25 > 35
  AND station_id = 's1'

Click a result row’s `pm25` cell. The eval strip’s `Condition on` text input auto-presets to the row’s provenance UUID, and the `Conditioned by:` badge underneath the input is active. Pick *Distribution profile* and run: the histogram now shows the *truncated* shape, restricted to the tail above `35`. Pick *Moment* with `k = 1` and `raw`: the panel returns the closed-form Mills-ratio mean of the [truncated normal](https://en.wikipedia.org/wiki/Truncated_normal_distribution), exactly $\mu + \sigma \cdot
\frac{\phi(\alpha)}{1 - \Phi(\alpha)}$ with $\alpha = (35 - \mu)/\sigma$. Click the active badge to clear the conditioning; the panel reverts to the unconditional mean $\mu$. Click the muted badge to restore the row provenance.

The closed-form truncation table covers Normal (Mills ratio), Uniform (intersected support), and Exponential (memorylessness on a lower bound or finite-interval truncation). For other shapes, the joint circuit between `pm25` and the row’s provenance is loaded with shared `gate_rv` leaves correctly coupled, and the conditional moment is estimated by rejection sampling at budget `provsql.rv_mc_samples`.

## Step 7: Diagnostic Sampling

For raw inspection or downstream analytics, draw samples from the conditional distribution. With the same row pinned and the *Conditioned by* badge active, pick *Sample* from the *Distribution* group; set `n = 200` and run. The result panel shows a six-value inline preview with a “show full list” expander; clicking it dumps all 200 samples.

For shapes that fall outside the closed-form table the sampler falls back to rejection sampling at the `provsql.rv_mc_samples` budget; if the conditioning event is so unlikely that fewer than `n` samples land inside that budget, the panel surfaces a hint pointing at the GUC, e.g. *Only 47 samples accepted within budget 10000; widen* `provsql.rv_mc_samples` *or loosen the conditioning*. Re-running with a larger budget (set `rv_mc_samples = 50000` in the Config panel) recovers the full batch.

## Step 8: Combining Batches via UNION

Both batches share the same id space (rows `1` through `8`, one per `(station, timestamp)` slot), so a `UNION` (without `ALL`) over `(station_id, id)` deduplicates a slot to a single result row whose provenance combines today’s reading and yesterday’s reading via the semiring addition. With `WHERE pm25 > 35` lifted on each branch, each contributing row carries a `gate_cmp(pm25 > 35)` of its own:

In [ ]:
(SELECT station_id, id FROM readings            WHERE pm25 > 35)
UNION
(SELECT station_id, id FROM historical_readings WHERE pm25 > 35)
ORDER BY station_id, id

Pick a result row’s `provsql` cell: Circuit mode shows a `gate_plus` (`⊕`) over the two contributing inputs (today’s row and yesterday’s row), each carrying its own `gate_cmp(pm25 > 35)` from the lifted `WHERE`. `probability_evaluate(provenance())` on the result gives the probability that *at least one* of the two days produced an Unhealthy reading for that slot. We deliberately keep the `random_variable` `pm25` column out of the `SELECT`: there is no duplicate-elimination semantics for `random_variable`, so a `UNION` over an RV column would have no well-defined meaning.

## Step 9: Filtering Grouped Random Variables by Expected Value

Filter the per-district aggregates from Step 5 by their expected average. Because [`avg`](https://provsql.org/doxygen-sql/html/group__random__variable__type.html#ga0243c3a6d53bdc36a89ce5131b38e504) over a `random_variable` column returns a `random_variable` (not an `agg_token`), and [`expected`](https://provsql.org/doxygen-sql/html/group__probability.html#ga362bcce6a7edb8e25174e64017ae9aba) collapses it to a plain `double`, the HAVING qual is deterministic from the planner-hook’s perspective; the rewrite leaves it for PostgreSQL to evaluate natively while still adding a `delta(gate_agg)` wrapper to each surviving group’s provenance:

In [ ]:
SELECT s.district, avg(r.pm25) AS avg_pm25
FROM readings r JOIN stations s ON s.id = r.station_id
GROUP BY s.district
HAVING expected(avg(r.pm25)) > 20

The inner [`avg`](https://provsql.org/doxygen-sql/html/group__random__variable__type.html#ga0243c3a6d53bdc36a89ce5131b38e504) is recognised as a `random_variable` aggregate (gate_arith DIV over per-row gate_mixture children, as in Step 5); [`expected`](https://provsql.org/doxygen-sql/html/group__probability.html#ga362bcce6a7edb8e25174e64017ae9aba) collapses the distribution to its mean (Monte Carlo here, since the DIV gate has no closed-form evaluator); the `> 20` is a plain comparison on a `double`, so the row survives iff its expected average exceeds the threshold. For the case-study fixture both districts pass (centre at ≈ 25.5, east at ≈ 21.6); clicking either result row’s `provsql` cell shows the `delta(gate_agg)` shape, identical to the no-HAVING aggregate from Step 5 but filtered to the surviving groups.

## Step 10: Independent vs Monte Carlo

For threshold queries whose contributing rows have structurally independent provenance, the `'independent'` probability method (see the chapter on probabilities) is *exact* and far cheaper than Monte Carlo. Compare the three available exact methods against `monte-carlo` on the Step 2 query:

In [ ]:
SELECT id,
  probability_evaluate(provenance(), 'independent')         AS p_ind,
  probability_evaluate(provenance(), 'monte-carlo', '10000') AS p_mc,
  probability_evaluate(provenance(), 'tree-decomposition')  AS p_td
FROM readings WHERE pm25 > 35

Studio’s eval strip exposes these methods directly; running each method against the same pinned subnode shows the analytic `independent` and `tree-decomposition` returning the same value to full precision, while `monte-carlo` returns a Hoeffding-bounded estimate that tightens as `n` grows.

See the chapter on continuous distributions for the full surface and the Studio chapter for the Studio reference.